In [1]:
from functions.nfl_clean_data import archive_files

In [3]:
import requests
from bs4 import BeautifulSoup as soup
import pandas as pd
import numpy as np
from functions.nfl_proj_scape import nfl_proj_scape

In [4]:
def get_spread_page():
    url = "https://rotogrinders.com/lineups/nfl"
    html = requests.get(url).text
    page = soup(html)
    return page
page = get_spread_page()

team_list = page.find_all("span", {"class": "shrt"})
teams = [i.text for i in team_list]
odds_list = page.find_all("a", {"href": "/nfl/odds"})
odds = [i.text for i in odds_list]
#teams.sort()
df_odds = dict(zip(teams,odds))
print(dict(sorted(df_odds.items(), key=lambda item: item[1])))


{'DET': '15.25', 'CHI': '15.25', 'HOU': '15.75', 'NYG': '16.75', 'JAC': '17.25', 'NYJ': '18.50', 'LVR': '19.25', 'ATL': '19.25', 'BAL': '20.25', 'PIT': '20.75', 'WAS': '21.25', 'CAR': '21.75', 'CLE': '23.25', 'CIN': '23.25', 'NOS': '24.00', 'MIN': '24.25', 'BUF': '24.25', 'LAR': '24.50', 'SEA': '25.25', 'SFO': '25.25', 'TEN': '26.25', 'LAC': '26.25', 'DEN': '26.75', 'ARI': '27.00', 'DAL': '27.25', 'TBB': '27.75', 'GBP': '27.75', 'KCC': '28.75'}


In [19]:
df_fd_temp = pd.read_csv('data/nfl/FanDuel-NFL-2021 ET-12 ET-12 ET-68179-players-list.csv')
df_fd = df_fd_temp.convert_dtypes()
df_fd.head()

,Id,Position,First Name,Nickname,Last Name,FPPG,Played,Salary,Game,Team,Opponent,Injury Indicator,Injury Details,Tier,Unnamed: 14,Unnamed: 15,Roster Position
0,68179-80001,RB,Austin,Austin Ekeler,Ekeler,19.341667,12,9200,NYG@LAC,LAC,NYG,<NA>,<NA>,<NA>,<NA>,<NA>,RB/FLEX
1,68179-42104,RB,Alvin,Alvin Kamara,Kamara,17.75,8,9000,NO@NYJ,NO,NYJ,<NA>,<NA>,<NA>,<NA>,<NA>,RB/FLEX
2,68179-62239,QB,Josh,Josh Allen,Allen,23.403333,12,8800,BUF@TB,BUF,TB,<NA>,<NA>,<NA>,<NA>,<NA>,QB
3,68179-53681,WR,Tyreek,Tyreek Hill,Hill,16.15,12,8700,LV@KC,KC,LV,<NA>,<NA>,<NA>,<NA>,<NA>,WR/FLEX
4,68179-57439,QB,Patrick,Patrick Mahomes,Mahomes,21.196667,12,8500,LV@KC,KC,LV,<NA>,<NA>,<NA>,<NA>,<NA>,QB


In [20]:
df_teams = list(df_fd.Team.unique())
df_teams.sort()
print(df_teams)

['ATL', 'BAL', 'BUF', 'CAR', 'CIN', 'CLE', 'DAL', 'DEN', 'DET', 'HOU', 'JAC', 'KC', 'LAC', 'LV', 'NO', 'NYG', 'NYJ', 'SEA', 'SF', 'TB', 'TEN', 'WAS']


In [21]:
replace_dict = {"GBP":"GB", "KCC":"KC", "LVR":"LV", "NEP":"NE", "NOS":"NO", "SFO":"SF", "TBB":"TB"}
clean_teams = [replace_dict.get(item,item) for item in teams]
print(clean_teams)

['PIT', 'MIN', 'BAL', 'CLE', 'DAL', 'WAS', 'LV', 'KC', 'SEA', 'HOU', 'JAC', 'TEN', 'NO', 'NYJ', 'ATL', 'CAR', 'DET', 'DEN', 'NYG', 'LAC', 'SF', 'CIN', 'BUF', 'TB', 'CHI', 'GB', 'LAR', 'ARI']


In [22]:
df_odds = dict(zip(clean_teams, odds))
#print(df_odds)
print(dict(sorted(df_odds.items(), key=lambda item: item[1])))

{'DET': '15.25', 'CHI': '15.25', 'HOU': '15.75', 'NYG': '16.75', 'JAC': '17.25', 'NYJ': '18.50', 'LV': '19.25', 'ATL': '19.25', 'BAL': '20.25', 'PIT': '20.75', 'WAS': '21.25', 'CAR': '21.75', 'CLE': '23.25', 'CIN': '23.25', 'NO': '24.00', 'MIN': '24.25', 'BUF': '24.25', 'LAR': '24.50', 'SEA': '25.25', 'SF': '25.25', 'TEN': '26.25', 'LAC': '26.25', 'DEN': '26.75', 'ARI': '27.00', 'DAL': '27.25', 'TB': '27.75', 'GB': '27.75', 'KC': '28.75'}


In [23]:
stats_df = nfl_proj_scape()
stats_df.head()

,Nickname,FP,Value
0,Josh Allen,22.0,2.49
1,Patrick Mahomes,21.6,2.54
2,Dak Prescott,21.0,2.59
3,Lamar Jackson,21.0,2.65
4,Justin Herbert,20.4,2.43


In [24]:
pd.set_option('display.max_rows', None)
df_fd_def = df_fd.query("Position == 'D'")
df_fd = df_fd.query("Position != 'D'")
df_fd = pd.merge(df_fd, stats_df, how='outer', on='Nickname')

In [25]:
df_fd.head(10)

,Id,Position,First Name,Nickname,Last Name,FPPG,Played,Salary,Game,Team,Opponent,Injury Indicator,Injury Details,Tier,Unnamed: 14,Unnamed: 15,Roster Position,FP,Value
0,68179-80001,RB,Austin,Austin Ekeler,Ekeler,19.341667,12,9200,NYG@LAC,LAC,NYG,<NA>,<NA>,<NA>,<NA>,<NA>,RB/FLEX,18.4,2
1,68179-42104,RB,Alvin,Alvin Kamara,Kamara,17.75,8,9000,NO@NYJ,NO,NYJ,<NA>,<NA>,<NA>,<NA>,<NA>,RB/FLEX,17.4,1.94
2,68179-62239,QB,Josh,Josh Allen,Allen,23.403333,12,8800,BUF@TB,BUF,TB,<NA>,<NA>,<NA>,<NA>,<NA>,QB,22.0,2.49
3,68179-53681,WR,Tyreek,Tyreek Hill,Hill,16.15,12,8700,LV@KC,KC,LV,<NA>,<NA>,<NA>,<NA>,<NA>,WR/FLEX,15.1,1.73
4,68179-57439,QB,Patrick,Patrick Mahomes,Mahomes,21.196667,12,8500,LV@KC,KC,LV,<NA>,<NA>,<NA>,<NA>,<NA>,QB,21.6,2.54
5,68179-52442,RB,Joe,Joe Mixon,Mixon,17.666667,12,8500,SF@CIN,CIN,SF,Q,Illness,<NA>,<NA>,<NA>,RB/FLEX,15.7,1.85
6,68179-56018,WR,Deebo,Deebo Samuel,Samuel,18.627272,11,8500,SF@CIN,SF,CIN,Q,Groin,<NA>,<NA>,<NA>,WR/FLEX,8.6,1.01
7,68179-69189,QB,Justin,Justin Herbert,Herbert,23.481667,12,8400,NYG@LAC,LAC,NYG,<NA>,<NA>,<NA>,<NA>,<NA>,QB,20.4,2.43
8,68179-6498,QB,Tom,Tom Brady,Brady,23.670001,12,8200,BUF@TB,TB,BUF,<NA>,<NA>,<NA>,<NA>,<NA>,QB,19.9,2.43
9,68179-25079,WR,Stefon,Stefon Diggs,Diggs,13.941667,12,8200,BUF@TB,BUF,TB,<NA>,<NA>,<NA>,<NA>,<NA>,WR/FLEX,14.7,1.79


In [26]:
df_fd.FPPG = df_fd.FP
df_fd.dropna(axis=0, subset=['Salary'], inplace=True)

In [27]:
df_fd['FPPG'] = df_fd.FPPG.astype(float)

In [28]:
df_fd.drop(axis=1, labels=['FP','Value'], inplace=True)

In [29]:
df_fd = df_fd.append(df_fd_def)
df_fd = df_fd[df_fd.FPPG > 5.0]
df_fd.columns

Index(['Id', 'Position', 'First Name', 'Nickname', 'Last Name', 'FPPG',
       'Played', 'Salary', 'Game', 'Team', 'Opponent', 'Injury Indicator',
       'Injury Details', 'Tier', 'Unnamed: 14', 'Unnamed: 15',
       'Roster Position'],
      dtype='object')

In [30]:
df_fd

,Id,Position,First Name,Nickname,Last Name,FPPG,Played,Salary,Game,Team,Opponent,Injury Indicator,Injury Details,Tier,Unnamed: 14,Unnamed: 15,Roster Position
0,68179-80001,RB,Austin,Austin Ekeler,Ekeler,18.4,12,9200,NYG@LAC,LAC,NYG,<NA>,<NA>,<NA>,<NA>,<NA>,RB/FLEX
1,68179-42104,RB,Alvin,Alvin Kamara,Kamara,17.4,8,9000,NO@NYJ,NO,NYJ,<NA>,<NA>,<NA>,<NA>,<NA>,RB/FLEX
2,68179-62239,QB,Josh,Josh Allen,Allen,22.0,12,8800,BUF@TB,BUF,TB,<NA>,<NA>,<NA>,<NA>,<NA>,QB
3,68179-53681,WR,Tyreek,Tyreek Hill,Hill,15.1,12,8700,LV@KC,KC,LV,<NA>,<NA>,<NA>,<NA>,<NA>,WR/FLEX
4,68179-57439,QB,Patrick,Patrick Mahomes,Mahomes,21.6,12,8500,LV@KC,KC,LV,<NA>,<NA>,<NA>,<NA>,<NA>,QB
5,68179-52442,RB,Joe,Joe Mixon,Mixon,15.7,12,8500,SF@CIN,CIN,SF,Q,Illness,<NA>,<NA>,<NA>,RB/FLEX
6,68179-56018,WR,Deebo,Deebo Samuel,Samuel,8.6,11,8500,SF@CIN,SF,CIN,Q,Groin,<NA>,<NA>,<NA>,WR/FLEX
7,68179-69189,QB,Justin,Justin Herbert,Herbert,20.4,12,8400,NYG@LAC,LAC,NYG,<NA>,<NA>,<NA>,<NA>,<NA>,QB
8,68179-6498,QB,Tom,Tom Brady,Brady,19.9,12,8200,BUF@TB,TB,BUF,<NA>,<NA>,<NA>,<NA>,<NA>,QB
9,68179-25079,WR,Stefon,Stefon Diggs,Diggs,14.7,12,8200,BUF@TB,BUF,TB,<NA>,<NA>,<NA>,<NA>,<NA>,WR/FLEX


In [31]:
df_fd.to_csv('data/nfl/FDprojections.csv')

In [37]:
from pydfs_lineup_optimizer import Site, Sport, get_optimizer, PlayerFilter, AfterEachExposureStrategy, TeamStack

optimizer = get_optimizer(Site.FANDUEL, Sport.FOOTBALL)
optimizer.load_players_from_csv("data/nfl/FDprojections.csv")
optimizer.restrict_positions_for_opposing_team(['D'], ['QB', 'RB', 'WR', 'TE', 'FLEX']),
optimizer.add_stack(TeamStack(2, for_positions=['QB', 'WR'], for_teams=['TB','KC','DAL']))
#optimizer.add_stack(TeamStack(3, for_teams=['TB', 'LAR', 'Cx_IN']))
for lineup in optimizer.optimize(10, max_exposure=0.6, exposure_strategy=AfterEachExposureStrategy):
    print(lineup)
optimizer.export('data/nfl/fd_result.csv')

/home/michael/.local/share/virtualenvs/fan_fact-icAuygf9/lib/python3.8/site-packages/pydfs_lineup_optimizer/utils.py:156: DeprecationWarning: available_teams will be removed in version 3.7, use player_pool.available_teams instead
  warnings.warn(text, DeprecationWarning)
/home/michael/.local/share/virtualenvs/fan_fact-icAuygf9/lib/python3.8/site-packages/pydfs_lineup_optimizer/utils.py:156: DeprecationWarning: available_teams will be removed in version 3.7, use player_pool.available_teams instead
  warnings.warn(text, DeprecationWarning)
/home/michael/.local/share/virtualenvs/fan_fact-icAuygf9/lib/python3.8/site-packages/pydfs_lineup_optimizer/utils.py:156: DeprecationWarning: available_teams will be removed in version 3.7, use player_pool.available_teams instead
  warnings.warn(text, DeprecationWarning)


 1. QB      Patrick Mahomes               QB    KC             LV@KC    21.6           8500.0$   
 2. RB      Javonte Williams              RB    DEN            DET@DEN  15.5           6700.0$   
 3. RB      Josh Jacobs                   RB    LV             LV@KC    15.2           7100.0$   
 4. WR      Ja'Marr Chase                 WR    CIN            SF@CIN   13.8           7200.0$   
 5. WR      Jamison Crowder               WR    NYJ            NO@NYJ   10.4           5500.0$   
 6. WR      Tyreek Hill                   WR    KC             LV@KC    15.1           8700.0$   
 7. TE      George Kittle                 TE    SF             SF@CIN   14.1           7100.0$   
 8. FLEX    Ty Johnson                    RB    NYJ            NO@NYJ   10.5           4900.0$   
 9. DEF     Dallas Cowboys                D     DAL            DAL@WAS  9.75           4200.0$   

Fantasy Points 125.95
Salary 59900.00

 1. QB      Tom Brady                     QB    TB             BUF@TB   19.9  